# 06 — SCL-NMA: Negator / Modal / Adverb Phrases

Loads the **SCL-NMA** lexicon — 3,207 modifier constructions where
a **negator**, **modal**, or **degree adverb** shifts the valence of a head word
(e.g. `not bad`, `barely good`, `might succeed`).

**Source:** `data/SCL-NMA/SCL-NMA/SCL-NMA.txt`  
**Modifier lists:** negators, modals, adverbs (same dir)  
**Scale:** bipolar valence [-1, +1]  
**Columns kept:** all originals (`term`, `nma_valence`) + modifier flags


In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

NMA_FILE = Path('data/SCL-NMA/SCL-NMA/SCL-NMA.txt')
NEG_FILE = Path('data/SCL-NMA/SCL-NMA/list-English-negators.txt')
MOD_FILE = Path('data/SCL-NMA/SCL-NMA/list-English-modals.txt')
ADV_FILE = Path('data/SCL-NMA/SCL-NMA/list-English-adverbs.txt')


## Load

In [2]:
nma = pd.read_csv(NMA_FILE, sep='\t', header=None, names=['term', 'nma_valence'])

negators = [l.strip() for l in NEG_FILE.read_text().splitlines() if l.strip()]
modals   = [l.strip() for l in MOD_FILE.read_text().splitlines() if l.strip()]
adverbs  = [l.strip() for l in ADV_FILE.read_text().splitlines() if l.strip()]

print(f'Entries   : {len(nma):,}')
print(f'Valence range: [{nma["nma_valence"].min():.3f}, {nma["nma_valence"].max():.3f}]')
print(f'Negators  : {len(negators)} — {negators}')
print(f'Modals    : {len(modals)} — {modals}')
print(f'Adverbs   : {len(adverbs)}')
nma.head(5)


Entries   : 3,207
Valence range: [-0.986, 0.986]
Negators  : 15 — ['cannot', 'could not', 'did not', 'does not', 'had no', 'have no', 'may not', 'never', 'no', 'not', 'nothing', 'was no', 'was not', 'will not', 'would not']
Modals    : 7 — ['can', 'could', 'may', 'might', 'must', 'should', 'would']
Adverbs   : 22


,term,nma_valence
0,enthusiastic,0.986
1,excellence,0.984
2,greatness,0.972
3,perfect,0.972
4,glorious,0.972


## Modifier-type flags

In [3]:
import re

def first_modifier(term, word_list):
    tokens = str(term).lower().split()
    for w in word_list:
        if re.search(rf'\\b{re.escape(w)}\\b', ' '.join(tokens)):
            return w
    return None

nma['modifier_negator'] = nma['term'].apply(lambda t: first_modifier(t, negators))
nma['modifier_modal']   = nma['term'].apply(lambda t: first_modifier(t, modals))
nma['modifier_adverb']  = nma['term'].apply(lambda t: first_modifier(t, adverbs))

has_neg = nma['modifier_negator'].notna()
has_mod = nma['modifier_modal'].notna()
has_adv = nma['modifier_adverb'].notna()
print(f'Has negator : {has_neg.sum():,}')
print(f'Has modal   : {has_mod.sum():,}')
print(f'Has adverb  : {has_adv.sum():,}')
nma.head(5)


Has negator : 0
Has modal   : 0
Has adverb  : 0


,term,nma_valence,modifier_negator,modifier_modal,modifier_adverb
0,enthusiastic,0.986,None,None,None
1,excellence,0.984,None,None,None
2,greatness,0.972,None,None,None
3,perfect,0.972,None,None,None
4,glorious,0.972,None,None,None


## Valence distribution

In [4]:
fig = px.histogram(nma, x='nma_valence', nbins=40,
    title='SCL-NMA valence distribution',
    labels={'nma_valence': 'Valence [-1, +1]'})
fig.add_vline(x=0, line_dash='dash', line_color='grey')
fig.show()

print(nma['nma_valence'].describe().round(4).to_string())
print(f'\nPositive (>0) : {(nma["nma_valence"] > 0).sum():,}')
print(f'Negative (<0) : {(nma["nma_valence"] < 0).sum():,}')
print(f'Neutral  (=0) : {(nma["nma_valence"] == 0).sum():,}')


count    3207.0000
mean       -0.0005
std         0.4941
min        -0.9860
25%        -0.4130
50%         0.0140
75%         0.4170
max         0.9860

Positive (>0) : 1,607
Negative (<0) : 1,575
Neutral  (=0) : 25


In [5]:
nma['modifier_type'] = (
    nma['modifier_negator'].notna().map({True:'negator',False:''})
    .where(nma['modifier_negator'].notna())
    .fillna(nma['modifier_modal'].notna().map({True:'modal',False:''}).where(nma['modifier_modal'].notna()))
    .fillna(nma['modifier_adverb'].notna().map({True:'adverb',False:''}).where(nma['modifier_adverb'].notna()))
    .fillna('other')
)
fig2 = px.violin(nma, x='modifier_type', y='nma_valence', box=True, points='outliers',
    title='SCL-NMA valence by modifier type',
    labels={'modifier_type':'Modifier type','nma_valence':'Valence'})
fig2.show()


## Most positive / most negative

In [6]:
print('Most positive:')
display(nma.nlargest(10, 'nma_valence')[['term','nma_valence','modifier_type']])
print('\nMost negative:')
display(nma.nsmallest(10, 'nma_valence')[['term','nma_valence','modifier_type']])


Most positive:


,term,nma_valence,modifier_type
0,enthusiastic,0.986,other
1,excellence,0.984,other
2,greatness,0.972,other
3,perfect,0.972,other
4,glorious,0.972,other
5,heavenly,0.958,other
6,excitement,0.958,other
7,joy,0.944,other
8,romantic,0.944,other
9,particularly happy,0.944,other



Most negative:


,term,nma_valence,modifier_type
3204,murder,-0.986,other
3205,terror,-0.986,other
3206,tragic,-0.986,other
3201,dying,-0.972,other
3202,menace,-0.972,other
3203,grave,-0.972,other
3196,repulse,-0.958,other
3197,criminal,-0.958,other
3198,repulsive,-0.958,other
3199,violent,-0.958,other
